In [ ]:
import os
import json
import glob
import pandas as pd


root_dir = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\physical\physical"


out_csv = os.path.join(root_dir, "train_physical_flat.csv")


MAX_FILES = 30



def load_json(path: str):
    """Load a JSON file safely."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def iter_records(obj):
    """
    Yield 1 or more 'records' from a JSON object.
    Handles common patterns:
      - list of records
      - dict with 'records' / 'data' / 'states' / 'trajectory'
      - dict with a single snapshot
    """
    if isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict):
                yield item
        return

    if not isinstance(obj, dict):
        return

   
    for key in ["records", "data", "states", "trajectory", "timesteps", "samples"]:
        if key in obj and isinstance(obj[key], list):
            for item in obj[key]:
                if isinstance(item, dict):
                    rec = dict(item)
                    # optionally carry top-level metadata into each record
                    for meta_key in ["scenario", "episode", "run_id", "seed", "file_id"]:
                        if meta_key in obj and meta_key not in rec:
                            rec[meta_key] = obj[meta_key]
                    yield rec
            return

   
    yield obj

def clean_label(series: pd.Series) -> pd.Series:
    """
    Convert labels to numeric in a tolerant way:
    - 'true'/'false' -> 1/0
    - strings containing digits -> extract digits
    - missing/unparseable -> 0
    """
    s = series.astype(str).str.strip()
    lower = s.str.lower()

    if lower.isin(["true", "false"]).all():
        return lower.map({"false": 0, "true": 1}).astype("int64")

    extracted = s.str.extract(r"(\d+)")[0]
    extracted = pd.to_numeric(extracted, errors="coerce").fillna(0)
    return extracted.astype("int64")



json_files = sorted(glob.glob(os.path.join(root_dir, "*.json")))
if not json_files:
    raise FileNotFoundError(f"No .json files found in: {root_dir}")

if MAX_FILES is not None:
    json_files = json_files[:MAX_FILES]

# Start fresh output
if os.path.exists(out_csv):
    os.remove(out_csv)

wrote_header = False
total_rows = 0

for i, path in enumerate(json_files, start=1):
    try:
        obj = load_json(path)
    except Exception as e:
        print(f"[SKIP] Failed to read {os.path.basename(path)}: {e}")
        continue

    records = list(iter_records(obj))
    if not records:
        print(f"[SKIP] No records found in {os.path.basename(path)}")
        continue


    df = pd.json_normalize(records)

    # Track provenance
    df["source_file"] = os.path.basename(path)


    if "state" in df.columns and df["state"].apply(lambda x: isinstance(x, dict)).any():
        state_df = pd.json_normalize(df["state"])
        state_df.columns = [f"state.{c}" for c in state_df.columns]
        df = pd.concat(
            [df.drop(columns=["state"]).reset_index(drop=True),
             state_df.reset_index(drop=True)],
            axis=1
        )

  
    for label_col in ["malicious", "attack", "label", "is_attack", "anomaly"]:
        if label_col in df.columns:
            df[label_col] = clean_label(df[label_col])


    def safe_to_numeric(col: pd.Series) -> pd.Series:
        try:
            return pd.to_numeric(col)
        except Exception:
            return col
    df = df.apply(safe_to_numeric)


   
    df.to_csv(out_csv, index=False, mode="a", header=not wrote_header)
    wrote_header = True

    total_rows += len(df)
    if i % 50 == 0:
        print(f"[OK] Processed {i}/{len(json_files)} files | rows so far: {total_rows}")

print("\nDONE ")
print("Files processed:", len(json_files))
print("Total rows written:", total_rows)
print("Output CSV:", out_csv)


df_all = pd.read_csv(out_csv)
print("\nFinal shape:", df_all.shape)
print("First 25 columns:", df_all.columns[:25].tolist())
print(df_all.head())


for col in ["malicious", "attack", "label", "is_attack", "anomaly"]:
    if col in df_all.columns:
        print(f"\n{col} distribution:")
        print(df_all[col].value_counts().head(20))



DONE ✅
Files processed: 30
Total rows written: 30
Output CSV: C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\physical\physical\train_physical_flat.csv

Final shape: (30, 788)
First 25 columns: ['timestamp', 'values.bus.0.CONFIGURATION.in_service', 'values.bus.0.MEASUREMENT.voltage', 'values.bus.0.MEASUREMENT.voltage_angle', 'values.bus.0.MEASUREMENT.active_power', 'values.bus.0.MEASUREMENT.reactive_power', 'values.bus.1.CONFIGURATION.in_service', 'values.bus.1.MEASUREMENT.voltage', 'values.bus.1.MEASUREMENT.voltage_angle', 'values.bus.1.MEASUREMENT.active_power', 'values.bus.1.MEASUREMENT.reactive_power', 'values.bus.2.CONFIGURATION.in_service', 'values.bus.2.MEASUREMENT.voltage', 'values.bus.2.MEASUREMENT.voltage_angle', 'values.bus.2.MEASUREMENT.active_power', 'values.bus.2.MEASUREMENT.reactive_power', 'values.bus.3.CONFIGURATION.in_service', 'values.bus.3.MEASUREMENT.voltage', 'values.bus.3.MEASUREMENT.voltage_angle', 'values.bus.3.MEASUREMENT.active_power', '

In [11]:
import re
import pandas as pd

phys_csv = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\physical\physical\train_physical_flat.csv"
phys = pd.read_csv(phys_csv)

def extract_ts_id(name: str):
    m = re.search(r"(TS\d+)", str(name))
    return m.group(1) if m else None

phys["ts_id"] = phys["source_file"].apply(extract_ts_id)

print(phys[["source_file", "ts_id"]].head(10))
print("\nts_id counts:")
print(phys["ts_id"].value_counts().head(20))


                                         source_file         ts_id
0  power_grid_2025-03-09T21-28-57-593260+00-00TS1...  TS1741555737
1  power_grid_2025-03-09T21-28-58-364693+00-00TS1...  TS1741555738
2  power_grid_2025-03-09T21-28-58-923532+00-00TS1...  TS1741555738
3  power_grid_2025-03-09T21-28-59-610097+00-00TS1...  TS1741555739
4  power_grid_2025-03-09T21-29-00-232237+00-00TS1...  TS1741555740
5  power_grid_2025-03-09T21-29-01-014088+00-00TS1...  TS1741555741
6  power_grid_2025-03-09T21-29-01-526928+00-00TS1...  TS1741555741
7  power_grid_2025-03-09T21-29-02-344683+00-00TS1...  TS1741555742
8  power_grid_2025-03-09T21-29-03-329643+00-00TS1...  TS1741555743
9  power_grid_2025-03-09T21-29-03-829920+00-00TS1...  TS1741555743

ts_id counts:
ts_id
TS1741555738    2
TS1741555741    2
TS1741555751    2
TS1741555748    2
TS1741555747    2
TS1741555743    2
TS1741555752    2
TS1741555757    2
TS1741555739    1
TS1741555737    1
TS1741555742    1
TS1741555740    1
TS1741555746    1
TS174155

In [12]:
import os, glob, json

train_root = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train"
keywords = {"malicious", "attack", "label", "anomaly", "is_attack", "event", "scenario", "intrusion", "alert"}

def get_keys(obj):
    if isinstance(obj, dict):
        return set(obj.keys())
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        return set(obj[0].keys())
    return set()

hits = []
json_paths = glob.glob(os.path.join(train_root, "**", "*.json"), recursive=True)

print("Total JSON files found:", len(json_paths))

for p in json_paths[:5000]:  # bump this up if needed
    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
        keys = {k.lower() for k in get_keys(obj)}
        if any(any(kw in k for k in keys) for kw in keywords):
            hits.append((p, sorted(list(keys))[:60]))
    except Exception:
        continue

print("\nPotential label-containing files (showing up to 20):")
for p, keys in hits[:20]:
    print("-", p)
    print("  keys(sample):", keys)


Total JSON files found: 37931

Potential label-containing files (showing up to 20):


In [13]:
import os, glob, json, re
import pandas as pd

train_root = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train"

def extract_ts_id(name: str):
    m = re.search(r"(TS\d+)", str(name))
    return m.group(1) if m else None

def find_label_in_obj(obj):
    """
    Returns (label_value, label_key_used) or (None, None) if not found.
    """
    if isinstance(obj, dict):
        # direct keys
        for k in ["malicious", "attack", "label", "is_attack", "anomaly"]:
            if k in obj:
                return obj[k], k

        # sometimes nested like {"meta": {...}}
        for container in ["meta", "metadata", "info", "summary"]:
            if container in obj and isinstance(obj[container], dict):
                for k in ["malicious", "attack", "label", "is_attack", "anomaly"]:
                    if k in obj[container]:
                        return obj[container][k], f"{container}.{k}"

        # events list
        if "events" in obj and isinstance(obj["events"], list):
            # try to detect an attack in any event entry
            for ev in obj["events"]:
                if isinstance(ev, dict):
                    for k in ["malicious", "attack", "label", "is_attack", "anomaly", "type", "name"]:
                        if k in ev:
                            return ev.get(k), f"events.{k}"

    # list of records case
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        # if every record has a label key, use the first
        for k in ["malicious", "attack", "label", "is_attack", "anomaly"]:
            if k in obj[0]:
                return obj[0][k], k

    return None, None

def clean_label_value(v):
    # bool -> 0/1
    if isinstance(v, bool):
        return int(v)
    # numbers -> int
    if isinstance(v, (int, float)) and v == v:
        return int(v)
    # strings -> try true/false or digits
    s = str(v).strip().lower()
    if s in ("true", "false"):
        return 1 if s == "true" else 0
    m = re.search(r"(\d+)", s)
    if m:
        return int(m.group(1))
    return None

rows = []
json_paths = glob.glob(os.path.join(train_root, "**", "*.json"), recursive=True)

for p in json_paths:
    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
        lab, key_used = find_label_in_obj(obj)
        lab_clean = clean_label_value(lab)

        if lab_clean is not None:
            rows.append({
                "source_file": os.path.basename(p),
                "ts_id": extract_ts_id(os.path.basename(p)),
                "label": lab_clean,
                "label_source_key": key_used,
                "label_path": p
            })
    except Exception:
        continue

labels = pd.DataFrame(rows)
print("Labels found:", len(labels))
print(labels.head(10))

labels_out = os.path.join(train_root, "labels_found.csv")
labels.to_csv(labels_out, index=False)
print("Saved:", labels_out)


Labels found: 0
Empty DataFrame
Columns: []
Index: []
Saved: C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\labels_found.csv


In [14]:
import json

path = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\control-center\control-center\ccx-notifications.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for i in range(5):
        line = f.readline()
        obj = json.loads(line)
        print("\nEVENT", i)
        print(obj)



EVENT 0
{'type': 'CONNECTION_CHANGE', 'data': {'protocol': '60870-5-104', 'server_ip': '172.16.2.2', 'server_port': 2404, 'connection_status': 'disconnected', 'server_key': 101}, 'timestamp': 1741555742.984146}

EVENT 1
{'type': 'CONNECTION_CHANGE', 'data': {'protocol': '60870-5-104', 'server_ip': '172.16.2.3', 'server_port': 2404, 'connection_status': 'disconnected', 'server_key': 102}, 'timestamp': 1741555742.989321}

EVENT 2
{'type': 'CONNECTION_CHANGE', 'data': {'protocol': '60870-5-104', 'server_ip': '172.16.2.4', 'server_port': 2404, 'connection_status': 'disconnected', 'server_key': 103}, 'timestamp': 1741555742.9961584}

EVENT 3
{'type': 'CONNECTION_CHANGE', 'data': {'protocol': '60870-5-104', 'server_ip': '172.16.2.5', 'server_port': 2404, 'connection_status': 'disconnected', 'server_key': 104}, 'timestamp': 1741555743.0027633}

EVENT 4
{'type': 'CONNECTION_CHANGE', 'data': {'protocol': '60870-5-104', 'server_ip': '172.16.2.6', 'server_port': 2404, 'connection_status': 'disco

In [15]:
import json
from collections import Counter

cc_path = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\control-center\control-center\ccx-notifications.jsonl"

type_counts = Counter()
N = 200000  # scan first 200k lines to start (increase later)

with open(cc_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N:
            break
        obj = json.loads(line)
        t = obj.get("type", "MISSING_TYPE")
        type_counts[t] += 1

print("Scanned lines:", sum(type_counts.values()))
print("Unique types:", len(type_counts))
print("\nTop 30 types:")
for t, c in type_counts.most_common(30):
    print(f"{t}: {c}")


Scanned lines: 200000
Unique types: 6

Top 30 types:
DATA_POINT_RECEIVED: 165040
RAW_PACKET_RECEIVED: 17567
RAW_PACKET_SENT: 17323
CONNECTION_CHANGE: 64
DATA_POINT_COMMAND_REPLY: 3
RESOLVE_ASYNC_RESPONSE: 3


In [16]:
import json
import pandas as pd

cc_path = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\control-center\control-center\ccx-notifications.jsonl"

# Fill these after you see the event types
ATTACK_TYPES = set([
    # example placeholders:
    # "INTRUSION_ALERT",
    # "ATTACK_DETECTED",
])

# If you don't know yet, you can start with "everything except connection changes"
IGNORE_TYPES = {"CONNECTION_CHANGE"}

labels = {}  # second -> 0/1

with open(cc_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        ts = obj.get("timestamp", None)
        t = obj.get("type", "")

        if ts is None:
            continue

        sec = int(float(ts))

        if ATTACK_TYPES:
            is_attack = int(t in ATTACK_TYPES)
        else:
            # temporary heuristic baseline: any non-connection-change event counts as suspicious
            is_attack = int(t not in IGNORE_TYPES)

        if is_attack:
            labels[sec] = 1
        else:
            labels.setdefault(sec, 0)

labels_df = pd.DataFrame({"timestamp_sec": list(labels.keys()),
                          "cc_label": list(labels.values())}).sort_values("timestamp_sec")

print(labels_df.head())
print("Seconds labeled:", len(labels_df))
print("Attack seconds:", labels_df["cc_label"].sum())

labels_out = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\control-center\control-center\cc_labels_per_second.csv"
labels_df.to_csv(labels_out, index=False)
print("Saved:", labels_out)


   timestamp_sec  cc_label
0     1741555742         0
1     1741555743         0
2     1741555753         1
3     1741555754         1
4     1741555755         1
Seconds labeled: 38090
Attack seconds: 38088
Saved: C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\control-center\control-center\cc_labels_per_second.csv


In [17]:
import pandas as pd

phys_csv = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\physical\physical\train_physical_flat.csv"
labels_csv = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\control-center\control-center\cc_labels_per_second.csv"

phys = pd.read_csv(phys_csv)
labels = pd.read_csv(labels_csv)

phys["timestamp_sec"] = phys["timestamp"].astype(float).astype(int)

merged = phys.merge(labels, on="timestamp_sec", how="left")
merged["cc_label"] = merged["cc_label"].fillna(0).astype(int)

print("Merged shape:", merged.shape)
print("Label counts:\n", merged["cc_label"].value_counts())

merged_out = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\train_physical_with_cc_labels.csv"
merged.to_csv(merged_out, index=False)
print("Saved:", merged_out)


Merged shape: (30, 790)
Label counts:
 cc_label
0    26
1     4
Name: count, dtype: int64
Saved: C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\train_physical_with_cc_labels.csv


In [18]:
import json
from collections import Counter

cc_path = r"C:\Users\amira\Downloads\SherlockDataset\01-Basic\01-Basic\raw\train\control-center\control-center\ccx-notifications.jsonl"

type_counts = Counter()

with open(cc_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        type_counts[obj.get("type", "MISSING")] += 1

print("Unique event types:", len(type_counts))
print("\nAll event types (sorted):\n")

for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"{t:40s} {c}")


Unique event types: 6

All event types (sorted):

DATA_POINT_RECEIVED                      1320955
RAW_PACKET_RECEIVED                      169257
RAW_PACKET_SENT                          138202
CONNECTION_CHANGE                        64
DATA_POINT_COMMAND_REPLY                 10
RESOLVE_ASYNC_RESPONSE                   10
